<a href="https://colab.research.google.com/github/vikrantyadav11234/NMT_OpenNMT_py/blob/main/OpenNMT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Gathering and Processing
To build a Machine Translation system, you need bilingual data, i.e. source sentences and their translations. You can use public bilingual corpora/datasets or you can use your translation memories (TMs). However, NMT requires a lot of data to train a good model, that is why most companies start with training a strong baseline model using public bilingual datasets, and then fine-tune this baseline model on their TMs. Sometimes also you can use pre-trained models directly for fine-tuning.

The majority of public bilingual datasets are collected on OPUS: https://opus.nlpl.eu/

Most of the datasets can be used for both commercial and non-commercial uses; however, some of them have more restricted licences. So you have to double-check the licence of a dataset before using it.

On OPUS, go to “Search & download resources” and choose two languages from the drop-down lists. You will see how it will list the available language datasets for this language pair. Try to use non-variant language codes like “en” for English and “fr” for French to get all the variants under this language. To know more details about a specific dataset, click its name.

In Machine Translation, we use the “Moses” format. Go ahead and try to download the “tico-19 v2020-10-28” by clicking “moses”. This will download a *.zip file; when you extract it, the two files that you care about are those whose names ending by the language codes. For example, for English to French, you will have “tico-19.en-fr.en” and “tico-19.en-fr.fr“. You can open these files with any text editor. Each file has a sentence/segment per line, and it is matching translation in the same line in the other file. This is what the "Moses" file format means.

Note that not all datasets are of the same quality. Some datasets have lower quality, especially big corpora crawled from the web. Check the provided “sample” before using the dataset. Nevertheless, even high-quality datasets, like those from the UN and EU, require filtering.

In [ ]:
# Create a directory and clone the Github MT-Preparation repository
!mkdir -p nmt_1
%cd nmt_1
!git clone https://github.com/ymoslem/MT-Preparation.git

/content/nmt_1
Cloning into 'MT-Preparation'...
remote: Enumerating objects: 272, done.
remote: Counting objects: 100% (272/272), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 272 (delta 135), reused 189 (delta 97), pack-reused 0
Receiving objects: 100% (272/272), 70.01 KiB | 4.67 MiB/s, done.
Resolving deltas: 100% (135/135), done.


In [ ]:
# Install the requirements
!pip3 install -r MT-Preparation/requirements.txt

In [ ]:
# Download and unzip a dataset
!wget https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-hi.txt.zip
!unzip en-hi.txt.zip

--2024-06-14 06:19:30--  https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-hi.txt.zip
Resolving object.pouta.csc.fi (object.pouta.csc.fi)... 86.50.254.18, 86.50.254.19
Connecting to object.pouta.csc.fi (object.pouta.csc.fi)|86.50.254.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 350169 (342K) [application/zip]
Saving to: ‘en-hi.txt.zip.1’

en-hi.txt.zip.1     100%[===================>] 341.96K   474KB/s    in 0.7s    

2024-06-14 06:19:32 (474 KB/s) - ‘en-hi.txt.zip.1’ saved [350169/350169]

Archive:  en-hi.txt.zip
  inflating: README                  
  inflating: LICENSE                 
  inflating: Tatoeba.en-hi.en        
  inflating: Tatoeba.en-hi.hi        
  inflating: Tatoeba.en-hi.xml       


# Data Filtering
Filtering out low-quality segments can help improve the translation quality of the output MT model. This might include misalignments, empty segments, duplicates, among other issues.

In [ ]:
!python3 MT-Preparation/filtering/filter.py Tatoeba.en-hi.en Tatoeba.en-hi.hi en hi

Dataframe shape (rows, columns): (11079, 2)
--- Rows with Empty Cells Deleted	--> Rows: 11072
--- Duplicates Deleted			--> Rows: 11071
--- Source-Copied Rows Deleted		--> Rows: 11071
--- Too Long Source/Target Deleted	--> Rows: 11054
--- HTML Removed			--> Rows: 11054
--- Rows will remain true-cased	--> Rows: 11054
--- Rows with Empty Cells Deleted	--> Rows: 11054
--- Rows Shuffled			--> Rows: 11054
--- Source Saved: Tatoeba.en-hi.en-filtered.en
--- Target Saved: Tatoeba.en-hi.hi-filtered.hi


In [ ]:
# Filter the dataset
# Arguments: source file, target file, source language, target language
!python3 MT-Preparation/filtering/filter.py UN.en-fr.fr UN.en-fr.en fr en

Dataframe shape (rows, columns): (74067, 2)
--- Rows with Empty Cells Deleted	--> Rows: 74067
--- Duplicates Deleted			--> Rows: 60662
--- Source-Copied Rows Deleted		--> Rows: 60476
--- Too Long Source/Target Deleted	--> Rows: 59719
--- HTML Removed			--> Rows: 59719
--- Rows will remain true-cased	--> Rows: 59719
--- Rows with Empty Cells Deleted	--> Rows: 59719
--- Rows Shuffled			--> Rows: 59719
--- Source Saved: UN.en-fr.fr-filtered.fr
--- Target Saved: UN.en-fr.en-filtered.en


# Tokenization / Sub-wording
To build a vocabulary for any NLP model, you have to tokenize (i.e. split) sentences into smaller units. Word-based tokenization used to be the way to go; in this case, each word would be a token. However, an MT model can only learn a specific number of vocabulary tokens due to limited hardware resources. To solve this issue, sub-words are used instead of whole words. At translation time, when the model sees a new word/token that looks like a word/token it has in the vocabulary, it still can try to continue the translation instead of marking this word as “unknown” or “unk”.

There are a few approaches to sub-wording such as BPE and the unigram model. One of the famous toolkits that incorporates the most common approaches is SentencePiece. Note that you have to train a sub-wording model and then use it. After translation, you will have to “desubword” or “decode” your text back using the same SentencePiece model.

In [ ]:
!ls MT-Preparation/subwording/

1-train_bpe.py	1-train_unigram.py  2-subword.py  3-desubword.py


In [ ]:
!ls MT-Preparation/subwording/

1-train_bpe.py	1-train_unigram.py  2-subword.py  3-desubword.py


In [ ]:
# eng to hindi
!python3 MT-Preparation/subwording/1-train_bpe.py Tatoeba.en-hi.en-filtered.en Tatoeba.en-hi.hi-filtered.hi

sentencepiece_trainer.cc(177) LOG(INFO) Running command: --input=Tatoeba.en-hi.en-filtered.en --model_prefix=source --vocab_size=50000 --hard_vocab_limit=false --model_type=bpe --split_digits=true
sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: Tatoeba.en-hi.en-filtered.en
  input_format: 
  model_prefix: source
  model_type: BPE
  vocab_size: 50000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  hard_vocab_limit: 0
  use_all_vocab: 0
  

In [ ]:
# Train a SentencePiece model for subword tokenization
!python3 MT-Preparation/subwording/1-train_unigram.py UN.en-fr.fr-filtered.fr UN.en-fr.en-filtered.en


sentencepiece_trainer.cc(177) LOG(INFO) Running command: --input=UN.en-fr.fr-filtered.fr --model_prefix=source --vocab_size=50000 --hard_vocab_limit=false --split_digits=true
sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: UN.en-fr.fr-filtered.fr
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 50000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
 

In [ ]:
# eng to hindi
!ls

en-hi.txt.zip	 README        target.vocab		     Tatoeba.en-hi.hi-filtered.hi
en-hi.txt.zip.1  source.model  Tatoeba.en-hi.en		     Tatoeba.en-hi.xml
LICENSE		 source.vocab  Tatoeba.en-hi.en-filtered.en
MT-Preparation	 target.model  Tatoeba.en-hi.hi


In [ ]:
!ls

en-fr.txt.zip	source.model  target.vocab	       UN.en-fr.fr
MT-Preparation	source.vocab  UN.en-fr.en	       UN.en-fr.fr-filtered.fr
README		target.model  UN.en-fr.en-filtered.en


In [ ]:
#eng to hindi
!python3 MT-Preparation/subwording/2-subword.py source.model target.model Tatoeba.en-hi.en-filtered.en Tatoeba.en-hi.hi-filtered.hi

Source Model: source.model
Target Model: target.model
Source Dataset: Tatoeba.en-hi.en-filtered.en
Target Dataset: Tatoeba.en-hi.hi-filtered.hi
Done subwording the source file! Output: Tatoeba.en-hi.en-filtered.en.subword
Done subwording the target file! Output: Tatoeba.en-hi.hi-filtered.hi.subword


In [ ]:
# Subword the dataset
!python3 MT-Preparation/subwording/2-subword.py source.model target.model UN.en-fr.fr-filtered.fr UN.en-fr.en-filtered.en


Source Model: source.model
Target Model: target.model
Source Dataset: UN.en-fr.fr-filtered.fr
Target Dataset: UN.en-fr.en-filtered.en
Done subwording the source file! Output: UN.en-fr.fr-filtered.fr.subword
Done subwording the target file! Output: UN.en-fr.en-filtered.en.subword


In [ ]:
#english to hindi
!head -n 3 Tatoeba.en-hi.en-filtered.en && echo "-----" && head -n 3 Tatoeba.en-hi.hi-filtered.hi

Don't ever forget this rule. 
I don't know. 
Did you love her? 
-----
यह नियम कभी मत भूलिएगा। 
मुझे नहीं मालूम। 
क्या तुम उनसे प्यार करते थे? 


In [ ]:
# First 3 lines before subwording
!head -n 3 UN.en-fr.fr-filtered.fr && echo "-----" && head -n 3 UN.en-fr.en-filtered.en


Adoptée à la 83e séance plénière, le 20 décembre 2006, sans avoir été mise aux voix, sur la recommandation de la Commission (A/61/424/Add.5, par. 8)Le projet de résolution recommandé dans le rapport de la Commission avait pour auteurs les pays suivants : Afrique du Sud, Allemagne, Arménie, Autriche, Azerbaïdjan, Bélarus, Belgique, Bosnie-Herzégovine, Bulgarie, Chili, Chypre, Croatie, Espagne, Estonie, États-Unis d'Amérique, Fédération de Russie, Finlande, France, Géorgie, Grèce, Hongrie, Irlande, Islande, Israël, Japon, Kazakhstan, Kirghizistan, Lettonie, Lituanie, Moldova, Mongolie, Ouzbékistan, Pologne, Portugal, République tchèque, Roumanie, Serbie, Slovaquie, Slovénie, Tadjikistan, Turkménistan, Turquie et Ukraine.
2. Demande au Secrétaire général de prendre les mesures voulues pour établir une coopération entre l'Organisation des Nations Unies et la Communauté économique des États d'Afrique centrale;
10. Décide de rationaliser les modalités d'établissement des rapports de l'Instit

In [ ]:
#english to hindi
!head -n 3 Tatoeba.en-hi.en-filtered.en.subword && echo "----" && head -n 3 Tatoeba.en-hi.hi-filtered.hi.subword

▁Don ' t ▁ever ▁forget ▁this ▁rule .
▁I ▁don ' t ▁know .
▁Did ▁you ▁love ▁her ?
----
▁यह ▁नियम ▁कभी ▁मत ▁भूलिएगा ।
▁मुझे ▁नहीं ▁मालूम ।
▁क्या ▁तुम ▁उनसे ▁प्यार ▁करते ▁थे ?


In [ ]:
# First 3 lines after subwording
!head -n 3 UN.en-fr.fr-filtered.fr.subword && echo "---" && head -n 3 UN.en-fr.en-filtered.en.subword


▁Adoptée ▁à ▁la ▁ 8 3 e ▁séance ▁plénière , ▁le ▁ 2 0 ▁décembre ▁ 2 0 0 6 , ▁sans ▁avoir ▁été ▁mise ▁aux ▁voix , ▁sur ▁la ▁recommandation ▁de ▁la ▁Commission ▁( A / 6 1 / 4 2 4 / Add . 5 , ▁par . ▁ 8 ) Le ▁projet ▁de ▁résolution ▁recommandé ▁d ans ▁le ▁rapport ▁de ▁la ▁Commission ▁avait ▁pour ▁auteurs ▁les ▁pays ▁suivants ▁: ▁Afrique ▁du ▁Sud , ▁Allemagne , ▁Arménie , ▁ Autriche , ▁Azerbaïdjan , ▁Bélarus , ▁Belg ique , ▁Bosnie - Herzégovine , ▁Bulgarie , ▁Chili , ▁Chypre , ▁Croatie , ▁Espagne , ▁Estonie , ▁États - Unis ▁d ' Amérique , ▁Fédération ▁de ▁Russie , ▁Finlande , ▁France , ▁Géorgie , ▁Grèce , ▁Hongrie , ▁Irlande , ▁Island e , ▁Isra ë l , ▁Japon , ▁Kazakhstan , ▁Kirghizistan , ▁Lettonie , ▁Lituanie , ▁Moldova , ▁Mongolie , ▁Ouzbékistan , ▁Pologne , ▁Portugal , ▁République ▁tchèque , ▁Roumanie , ▁Serbie , ▁Slovaquie , ▁Slovénie , ▁Tadjikistan , ▁Turkménistan , ▁Turquie ▁et ▁Ukraine .
▁ 2 . ▁Demande ▁au ▁Secrétaire ▁général ▁de ▁prendre ▁les ▁mesures ▁voulues ▁pour ▁établir ▁une 

# Data Splitting
We usually split our dataset into 3 portions:

training dataset - used for training the model;
development dataset - used to run regular validations during the training to help improve the model parameters; and
testing dataset - a holdout dataset used after the model finishes training to finally evaluate the model on unseen data.

In [ ]:
# Split the dataset into training set, development set, and test set
# Development and test sets should be between 1000 and 5000 segments (here we chose 2000)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 UN.en-fr.fr-filtered.fr.subword UN.en-fr.en-filtered.en.subword


Dataframe shape: (59719, 2)
--- Empty Cells Deleted --> Rows: 59719
--- Wrote Files
Done!
Output files
UN.en-fr.fr-filtered.fr.subword.train
UN.en-fr.en-filtered.en.subword.train
UN.en-fr.fr-filtered.fr.subword.dev
UN.en-fr.en-filtered.en.subword.dev
UN.en-fr.fr-filtered.fr.subword.test
UN.en-fr.en-filtered.en.subword.test


In [ ]:
# splitting- english to hindi
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 Tatoeba.en-hi.en-filtered.en.subword Tatoeba.en-hi.hi-filtered.hi.subword

Dataframe shape: (11054, 2)
--- Empty Cells Deleted --> Rows: 11054
--- Wrote Files
Done!
Output files
Tatoeba.en-hi.en-filtered.en.subword.train
Tatoeba.en-hi.hi-filtered.hi.subword.train
Tatoeba.en-hi.en-filtered.en.subword.dev
Tatoeba.en-hi.hi-filtered.hi.subword.dev
Tatoeba.en-hi.en-filtered.en.subword.test
Tatoeba.en-hi.hi-filtered.hi.subword.test


In [ ]:
# Line count for the subworded train, dev, test datatest
!wc -l *.subword.*

    2000 UN.en-fr.en-filtered.en.subword.dev
    2000 UN.en-fr.en-filtered.en.subword.test
   55719 UN.en-fr.en-filtered.en.subword.train
    2000 UN.en-fr.fr-filtered.fr.subword.dev
    2000 UN.en-fr.fr-filtered.fr.subword.test
   55719 UN.en-fr.fr-filtered.fr.subword.train
  119438 total


In [ ]:
# Line count- english to hindi
!wc -l *.subword.*

   2000 Tatoeba.en-hi.en-filtered.en.subword.dev
   2000 Tatoeba.en-hi.en-filtered.en.subword.test
   7054 Tatoeba.en-hi.en-filtered.en.subword.train
   2000 Tatoeba.en-hi.hi-filtered.hi.subword.dev
   2000 Tatoeba.en-hi.hi-filtered.hi.subword.test
   7054 Tatoeba.en-hi.hi-filtered.hi.subword.train
  22108 total


In [ ]:
# Check the first and last line from each dataset

# -------------------------------------------
# Change this cell to print your name
!echo -e "My name is: FirstName SecondName \n"
# -------------------------------------------

!echo "---First line---"
!head -n 1 *.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 *.{train,dev,test}

My name is: FirstName SecondName 

---First line---
==> UN.en-fr.en-filtered.en.subword.train <==
▁Adopt ed ▁at ▁the ▁ 8 3 rd ▁plenary ▁meeting , ▁on ▁ 2 0 ▁December ▁ 2 0 0 6 , ▁with out ▁a ▁vote , ▁on ▁the ▁recommendation ▁of ▁the ▁Committee ▁( A / 6 1 / 4 2 4 / Add . 5 , ▁para . ▁ 8 ) The ▁draft ▁resolution ▁recommended ▁in ▁the ▁report ▁was ▁ sponsored ▁in ▁the ▁Committee ▁by : ▁Armenia , ▁Austria , ▁Azerbaijan , ▁Belarus , ▁Belgium , ▁Bosnia ▁and ▁Herzegovina , ▁Bulgaria , ▁Chile , ▁Croatia , ▁Cyprus , ▁C ze ch ▁Republic , ▁Estonia , ▁Finland , ▁France , ▁Georgia , ▁Germany , ▁Greece , ▁Hungary , ▁Iceland , ▁Ireland , ▁Israel , ▁Japan , ▁Kazakhstan , ▁Kyrgyzstan , ▁Latvia , ▁Lithuania , ▁Moldova , ▁Mongolia , ▁Poland , ▁Portugal , ▁Romania , ▁Russian ▁Federation , ▁Serbia , ▁Slovakia , ▁Slovenia , ▁South ▁Africa , ▁Spain , ▁Tajikistan , ▁Turkey , ▁Turkmenistan , ▁Ukraine , ▁Unit ed ▁States ▁of ▁America ▁and ▁Uzbekistan .

==> UN.en-fr.fr-filtered.fr.subword.train <==
▁Adoptée ▁à ▁

In [ ]:
# english to hindi
# Check the first and last line from each dataset

# -------------------------------------------
# Change this cell to print your name
!echo -e "My name is: FirstName SecondName \n"
# -------------------------------------------

!echo "---First line---"
!head -n 1 *.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 *.{train,dev,test}

My name is: FirstName SecondName 

---First line---
==> Tatoeba.en-hi.en-filtered.en.subword.train <==
▁I ▁don ' t ▁know .

==> Tatoeba.en-hi.hi-filtered.hi.subword.train <==
▁मुझे ▁नहीं ▁मालूम ।

==> Tatoeba.en-hi.en-filtered.en.subword.dev <==
▁You ▁can ' t ▁let ▁that ▁come ▁between ▁you ▁and ▁Tom .

==> Tatoeba.en-hi.hi-filtered.hi.subword.dev <==
▁तुम ▁उस ▁बात ▁को ▁तुम्हारे ▁और ▁टॉम ▁के ▁बीच ▁आने ▁नहीं ▁दे ▁सकती ।

==> Tatoeba.en-hi.en-filtered.en.subword.test <==
▁We ▁can ' t ▁waste ▁any ▁more ▁time .

==> Tatoeba.en-hi.hi-filtered.hi.subword.test <==
▁हम ▁और ▁वक्त ▁बरबाद ▁नहीं ▁कर ▁सकते ।

---Last line---
==> Tatoeba.en-hi.en-filtered.en.subword.train <==
▁He ▁spoke ▁to ▁farmers ▁in ▁Iowa .

==> Tatoeba.en-hi.hi-filtered.hi.subword.train <==
▁उन्होंने ▁आयोवा ▁में ▁किसानों ▁से ▁बात ▁की ।

==> Tatoeba.en-hi.en-filtered.en.subword.dev <==
▁You ▁will ▁like ▁Germany .

==> Tatoeba.en-hi.hi-filtered.hi.subword.dev <==
▁आपको ▁जर्मनी ▁पसंद ▁आयेगा ।

==> Tatoeba.en-hi.en-filtered.en.subwo

In [ ]:

# Copy your data to your Google Drive
!cp -R /content/nmt/ /content/drive/MyDrive/

In [ ]:
# eng to hindi
# copy the data to google drive
!cp -R /content/nmt_1/ /content/drive/MyDrive/

In [ ]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

  Using cached OpenNMT_py-3.5.1-py3-none-any.whl (262 kB)
  Using cached torch-2.2.2-cp310-cp310-manylinux1_x86_64.whl (755.5 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.7.15 requires torchvision>=0.11, which is not installed.


In [ ]:

# Open the folder where you saved your prepapred datasets
# You might need to mount your Google Drive first
%cd /content/drive/MyDrive/nmt/
!ls

/content/drive/MyDrive/nmt
en-fr.txt.zip	UN.en-fr.en			       UN.en-fr.fr-filtered.fr
MT-Preparation	UN.en-fr.en-filtered.en		       UN.en-fr.fr-filtered.fr.subword
README		UN.en-fr.en-filtered.en.subword        UN.en-fr.fr-filtered.fr.subword.dev
source.model	UN.en-fr.en-filtered.en.subword.dev    UN.en-fr.fr-filtered.fr.subword.test
source.vocab	UN.en-fr.en-filtered.en.subword.test   UN.en-fr.fr-filtered.fr.subword.train
target.model	UN.en-fr.en-filtered.en.subword.train
target.vocab	UN.en-fr.fr


In [ ]:
# eng to hindi
%cd /content/drive/MyDrive/nmt_1//
!ls

/content/drive/MyDrive/nmt_1
en-hi.txt.zip		      Tatoeba.en-hi.en-filtered.en.subword
en-hi.txt.zip.1		      Tatoeba.en-hi.en-filtered.en.subword.dev
LICENSE			      Tatoeba.en-hi.en-filtered.en.subword.test
MT-Preparation		      Tatoeba.en-hi.en-filtered.en.subword.train
README			      Tatoeba.en-hi.hi
source.model		      Tatoeba.en-hi.hi-filtered.hi
source.vocab		      Tatoeba.en-hi.hi-filtered.hi.subword
target.model		      Tatoeba.en-hi.hi-filtered.hi.subword.dev
target.vocab		      Tatoeba.en-hi.hi-filtered.hi.subword.test
Tatoeba.en-hi.en	      Tatoeba.en-hi.hi-filtered.hi.subword.train
Tatoeba.en-hi.en-filtered.en  Tatoeba.en-hi.xml


# Create the Training Configuration File
The following config file matches most of the recommended values for the Transformer model Vaswani et al., 2017. As the current dataset is small, we reduced the following values:

* train_steps - for datasets with a few millions of sentences, consider using a value between 100000 and 200000, or more! Enabling the option early_stopping can help stop the training when there is no considerable improvement.
* valid_steps - 10000 can be good if the value train_steps is big enough.
warmup_steps - obviously, its value must be less than train_steps. Try 4000 and 8000 values.

In [ ]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: UN.en-fr.fr-filtered.fr.subword.train
        path_tgt: UN.en-fr.en-filtered.en.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: UN.en-fr.fr-filtered.fr.subword.dev
        path_tgt: UN.en-fr.en-filtered.en.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 50000
tgt_vocab_size: 50000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 150
src_seq_length: 150

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 10000

# To save space, limit checkpoints to last n
# keep_checkpoint: 10

seed: 3435

# Default: 100000 - Train the model to max n steps
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 3000

# Default: 10000 - Run validation after n steps
valid_steps: 1000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 1000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)


In [ ]:
# english to hindi


In [ ]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: Tatoeba.en-hi.en-filtered.en.subword.train
        path_tgt: Tatoeba.en-hi.hi-filtered.hi.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: Tatoeba.en-hi.en-filtered.en.subword.dev
        path_tgt: Tatoeba.en-hi.hi-filtered.hi.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 50000
tgt_vocab_size: 50000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 150
src_seq_length: 150

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.hien

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 10000

# To save space, limit checkpoints to last n
# keep_checkpoint: 10

seed: 3435

# Default: 100000 - Train the model to max n steps
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 3000

# Default: 10000 - Run validation after n steps
valid_steps: 1000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 1000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)


In [ ]:
# Check the content of the configuration file
!cat config.yaml


# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: UN.en-fr.fr-filtered.fr.subword.train
        path_tgt: UN.en-fr.en-filtered.en.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: UN.en-fr.fr-filtered.fr.subword.dev
        path_tgt: UN.en-fr.en-filtered.en.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 50000
tgt_vocab_size: 50000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 150
src_seq_length: 150

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n validations
early_st

In [ ]:
# english to hindi
# Check the content of the configuration file
!cat config.yaml

# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: Tatoeba.en-hi.en-filtered.en.subword.train
        path_tgt: Tatoeba.en-hi.hi-filtered.hi.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: Tatoeba.en-hi.en-filtered.en.subword.dev
        path_tgt: Tatoeba.en-hi.hi-filtered.hi.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 50000
tgt_vocab_size: 50000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 150
src_seq_length: 150

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n 

# Build Vocabulary
For large datasets, it is not feasable to use all words/tokens found in the corpus. Instead, a specific set of vocabulary is extracted from the training dataset, usually betweeen 32k and 100k words. This is the main purpose of the vocabulary building step.

In [ ]:
# Find the number of CPUs/cores on the machine
!nproc --all

2


In [ ]:
# eng to hindi
!nproc --all

2


In [ ]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 58.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 113.0 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  U

In [ ]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 2

Corpus corpus_1's weight should be given. We default it to 1 for you.
[2024-06-14 06:48:38,418 INFO] Counter vocab from -1 samples.
[2024-06-14 06:48:38,418 INFO] n_sample=-1: Build vocab on full datasets.
[2024-06-14 06:48:38,719 INFO] Counters src: 4458
[2024-06-14 06:48:38,719 INFO] Counters tgt: 5349


In [ ]:
# english to hindi
!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 2

/bin/bash: line 1: onmt_build_vocab: command not found


In [ ]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-af953f9b-a31b-8b7a-950c-1b98664b3ec0)


In [ ]:
# Check if the GPU is visable to PyTorch

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)

True
Tesla T4
Free GPU memory: 14999.0625 out of: 15102.0625


# Training
Now, start training your NMT model! 🎉 🎉 🎉

In [ ]:

!rm -rf drive/MyDrive/nmt/models/

In [ ]:
# english to hindi
!rm -rf drive/MyDrive/nmt_1/models/

In [ ]:
# Train the NMT model
!onmt_train -config config.yaml

Streaming output truncated to the last 5000 lines.
[2024-06-14 07:01:01,695 INFO] Weighted corpora loaded so far:
			* corpus_1: 1108
[2024-06-14 07:01:01,797 INFO] Weighted corpora loaded so far:
			* corpus_1: 1109
[2024-06-14 07:01:01,888 INFO] Weighted corpora loaded so far:
			* corpus_1: 1110
[2024-06-14 07:01:01,999 INFO] Weighted corpora loaded so far:
			* corpus_1: 1111
[2024-06-14 07:01:02,102 INFO] Weighted corpora loaded so far:
			* corpus_1: 1112
[2024-06-14 07:01:05,037 INFO] Weighted corpora loaded so far:
			* corpus_1: 1113
[2024-06-14 07:01:05,133 INFO] Weighted corpora loaded so far:
			* corpus_1: 1114
[2024-06-14 07:01:05,223 INFO] Weighted corpora loaded so far:
			* corpus_1: 1115
[2024-06-14 07:01:17,464 INFO] Weighted corpora loaded so far:
			* corpus_1: 1116
[2024-06-14 07:01:17,572 INFO] Weighted corpora loaded so far:
			* corpus_1: 1117
[2024-06-14 07:01:17,676 INFO] Weighted corpora loaded so far:
			* corpus_1: 1118
[2024-06-14 07:01:17,778 INFO] Weigh

In [ ]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
!onmt_translate -model models/model.fren_step_3000.pt -src UN.en-fr.fr-filtered.fr.subword.test -output UN.en.translated -gpu 0 -min_length 1


[2024-06-14 05:25:45,340 INFO] Loading checkpoint from models/model.fren_step_3000.pt
[2024-06-14 05:25:48,796 INFO] Loading data into the model
[2024-06-14 05:27:20,031 INFO] PRED SCORE: -0.2026, PRED PPL: 1.22 NB SENTENCES: 2000
Time w/o python interpreter load/terminate:  94.71984100341797


In [ ]:
# english to hindi
!onmt_translate -model models/model.hien

In [ ]:
# Check the first 5 lines of the translation file
!head -n 5 UN.en.translated

▁ 1 0 . ▁Requests ▁all ▁relevant ▁organs ▁and ▁agencies ▁of ▁the ▁Unit ed ▁Nations ▁system , ▁within ▁their ▁mandates , ▁to ▁provide ▁all ▁possible ▁assistance ▁and ▁support ▁to ▁the ▁Special ▁Representative ▁in ▁the ▁implementation ▁of ▁her ▁programme ▁of ▁activities ;
▁( d ) ▁To ▁request ▁the ▁Special ▁Representative ▁of ▁the ▁Secretary - General ▁for ▁Children ▁and ▁Arm ed ▁Conflict ▁to ▁submit ▁reports ▁to ▁the ▁General ▁Assembly ▁and ▁to ▁the ▁Commission ▁on ▁Human ▁Rights ▁reports ▁providing ▁relevant ▁information ▁on ▁the ▁situation ▁of ▁children ▁affected ▁by ▁ armed ▁conflict , ▁taking ▁into ▁account ▁the ▁outcome ▁document ▁adopted ▁by ▁the ▁General ▁Assembly ▁at ▁its ▁special ▁session ▁on ▁children ▁and ▁the ▁reports ▁of ▁relevant ▁bodies ;
▁ 1 . ▁Decides ▁to ▁invite ▁the ▁Shanghai ▁Cooperation ▁Organization ▁to ▁participate ▁in ▁its ▁sessions ▁and ▁the ▁work ▁of ▁the ▁General ▁Assembly ▁in ▁the ▁capacity ▁of ▁observer ;
▁RESOLUTION ▁ 5 6 / 5 6
▁ 8 . ▁Notes ▁with ▁deep ▁conc

In [ ]:

# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.4 MB/s eta 0:00:00


In [ ]:
# Desubword the translation file
!python3 MT-Preparation/subwording/3-desubword.py target.model UN.en.translated

Done desubwording! Output: UN.en.translated.desubword


In [ ]:
# Check the first 5 lines of the desubworded translation file
!head -n 5 UN.en.translated.desubword

10. Requests all relevant organs and agencies of the United Nations system, within their mandates, to provide all possible assistance and support to the Special Representative in the implementation of her programme of activities;
(d) To request the Special Representative of the Secretary-General for Children and Armed Conflict to submit reports to the General Assembly and to the Commission on Human Rights reports providing relevant information on the situation of children affected by armed conflict, taking into account the outcome document adopted by the General Assembly at its special session on children and the reports of relevant bodies;
1. Decides to invite the Shanghai Cooperation Organization to participate in its sessions and the work of the General Assembly in the capacity of observer;
RESOLUTION 56/56
8. Notes with deep concern the prison conditions in Cambodia, notes with interest some important efforts to improve the prison system, recommends the continuation of internationa

In [ ]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation,
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 MT-Preparation/subwording/3-desubword.py target.model UN.en-fr.en-filtered.en.subword.test


Done desubwording! Output: UN.en-fr.en-filtered.en.subword.test.desubword


In [ ]:
# Check the first 5 lines of the desubworded reference
!head -n 5 UN.en-fr.en-filtered.en.subword.test.desubword

10. Requests all concerned United Nations agencies and organizations within their mandates to provide all possible assistance and support to the Special Representative in the implementation of her programme of activities;
(d) To request the Special Representative of the Secretary-General for Children and Armed Conflict to submit to the General Assembly and the Commission on Human Rights reports containing relevant information on the situation of children affected by armed conflict, taking into account the outcome document adopted by the General Assembly at its special session on children and bearing in mind existing mandates and reports of relevant bodies;
1. Decides to invite the Shanghai Cooperation Organization to participate in the sessions and the work of the General Assembly in the capacity of observer;
RESOLUTION 56/56
8. Notes with serious concern the prison conditions in Cambodia, notes with interest some important efforts to improve the prison system, recommends the continuat

# MT Evaluation
There are several MT Evaluation metrics such as BLEU, TER, METEOR, COMET, BERTScore, among others.

Here we are using BLEU. Files must be detokenized/desubworded beforehand.

In [ ]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py


--2024-06-14 05:31:27--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py’

compute-bleu.py     100%[===================>]     957  --.-KB/s    in 0s      

2024-06-14 05:31:27 (17.1 MB/s) - ‘compute-bleu.py’ saved [957/957]



In [ ]:
# Install sacrebleu
!pip3 install sacrebleu

In [ ]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py UN.en-fr.en-filtered.en.subword.test.desubword UN.en.translated.desubword


Reference 1st sentence: 10. Requests all concerned United Nations agencies and organizations within their mandates to provide all possible assistance and support to the Special Representative in the implementation of her programme of activities;
MTed 1st sentence: 10. Requests all relevant organs and agencies of the United Nations system, within their mandates, to provide all possible assistance and support to the Special Representative in the implementation of her programme of activities;
BLEU:  65.83152325220466
